# 🚀 Phase 1 — The Two-Body Problem
## Orbital Mechanics & Astrodynamics — From First Principles to Simulation

> **Goal:** Derive, implement, and *validate* a physically correct orbital propagator from scratch  
> using only NumPy, then visualize and export results.

**After this notebook you will:**
- Understand the two-body EOM and why it reduces to a 6-DOF problem
- Have a working RK4 integrator written from scratch
- Know how to verify numerical accuracy using conservation laws
- Be able to simulate any Keplerian orbit and export results

**Read alongside:** `theory_01_two_body.md` for full mathematical derivations.


---
## 1. Setup & Physical Constants

We work in **km / s / kg** throughout — the standard in astrodynamics.

> ⚠️ Never mix SI meters with km. A 1000x error in $\mu$ is the #1 beginner mistake.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import csv, json, os
from datetime import datetime

# Physical constants (km, s, kg system)
MU_EARTH = 398_600.4418       # km^3/s^2
MU_SUN   = 1.327_124_4e11     # km^3/s^2
MU_MOON  = 4_902.800_066      # km^3/s^2
MU_MARS  = 42_828.375         # km^3/s^2
R_EARTH  = 6_371.0            # km
AU       = 1.495_978_707e8    # km

print("Constants loaded.")
print(f"  mu_Earth = {MU_EARTH:,.4f}  km3/s2")
print(f"  mu_Sun   = {MU_SUN:.6e}  km3/s2")
print(f"  R_Earth  = {R_EARTH:,.1f}  km")
print(f"  1 AU     = {AU:.6e}  km")


---
## 2. Equations of Motion

The **two-body EOM** in state-space form. We stack position and velocity into a 6-vector:

$$\vec{y} = \begin{pmatrix}\vec{r} \\ \vec{v}\end{pmatrix} \in \mathbb{R}^6$$

$$\dot{\vec{y}} = \begin{pmatrix}\vec{v} \\ -\dfrac{\mu}{r^3}\,\vec{r}\end{pmatrix}$$

The only nonlinearity is $r^{-3}$. The acceleration is always directed toward the primary (origin),  
and its magnitude follows the inverse-square law: $|\ddot{r}| = \mu/r^2$.


In [ ]:
def two_body_eom(t, state, mu):
    r_vec = state[:3]
    v_vec = state[3:]
    r_mag = np.linalg.norm(r_vec)
    if r_mag < 1e-10:
        raise ValueError("Singularity: r ~ 0. Bodies have collided.")
    accel = -(mu / r_mag**3) * r_vec
    return np.concatenate([v_vec, accel])

# Sanity check: surface gravity
r_surf  = np.array([R_EARTH, 0.0, 0.0])
v_zero  = np.zeros(3)
dydt    = two_body_eom(0, np.concatenate([r_surf, v_zero]), MU_EARTH)
g_surf  = np.linalg.norm(dydt[3:]) * 1000   # convert to m/s^2
print(f"Surface gravity: {g_surf:.4f} m/s^2  (expect ~9.82 m/s^2)")


---
## 3. RK4 Integrator — Built From Scratch

The 4th-order Runge-Kutta method estimates the next state by sampling the derivative  
at 4 points within each step and combining them with Simpson-like weights:

$$k_1 = f(t_n,\ y_n)$$
$$k_2 = f\!\left(t_n + \tfrac{h}{2},\ y_n + \tfrac{h}{2}k_1\right)$$
$$k_3 = f\!\left(t_n + \tfrac{h}{2},\ y_n + \tfrac{h}{2}k_2\right)$$
$$k_4 = f(t_n + h,\ y_n + h\,k_3)$$

$$\boxed{y_{n+1} = y_n + \frac{h}{6}(k_1 + 2k_2 + 2k_3 + k_4)}$$

**Global error $O(h^4)$:** halving step size reduces error by 16x.


In [ ]:
def rk4_step(f, t, y, h, *args):
    k1 = f(t,         y,               *args)
    k2 = f(t + h/2,   y + (h/2)*k1,   *args)
    k3 = f(t + h/2,   y + (h/2)*k2,   *args)
    k4 = f(t + h,     y + h*k3,        *args)
    return y + (h/6.0) * (k1 + 2*k2 + 2*k3 + k4)


def propagate(eom, state0, t_span, dt, mu):
    t0, tf    = t_span
    n_steps   = int((tf - t0) / dt)
    t_arr     = np.zeros(n_steps + 1)
    states    = np.zeros((n_steps + 1, 6))
    t_arr[0]  = t0
    states[0] = state0
    for i in range(n_steps):
        states[i+1] = rk4_step(eom, t_arr[i], states[i], dt, mu)
        t_arr[i+1]  = t_arr[i] + dt
    return {"t": t_arr, "states": states,
            "r": states[:, :3], "v": states[:, 3:]}

print("RK4 integrator ready.")


---
## 4. Orbital Mechanics Helper Functions

### Key conserved quantities:
- **Specific energy:** $\varepsilon = v^2/2 - \mu/r$  (constant for two-body)
- **Specific angular momentum:** $\vec{h} = \vec{r} \times \vec{v}$  (constant vector)
- **Eccentricity vector:** $\vec{e} = \frac{\vec{v}\times\vec{h}}{\mu} - \hat{r}$ (points to periapsis)

### Kepler's Third Law:
$$T = 2\pi\sqrt{a^3/\mu}$$


In [ ]:
def specific_energy(r_vec, v_vec, mu):
    return 0.5*np.linalg.norm(v_vec)**2 - mu/np.linalg.norm(r_vec)

def specific_ang_mom(r_vec, v_vec):
    return np.cross(r_vec, v_vec)

def ecc_vector(r_vec, v_vec, mu):
    r, v = np.linalg.norm(r_vec), np.linalg.norm(v_vec)
    return (1/mu)*((v**2 - mu/r)*r_vec - np.dot(r_vec, v_vec)*v_vec)

def orbital_period(a, mu):
    return 2*np.pi*np.sqrt(a**3/mu)

def circular_orbit_state(alt_km, inc_deg=0.0, mu=MU_EARTH):
    r   = R_EARTH + alt_km
    v_c = np.sqrt(mu/r)
    i   = np.radians(inc_deg)
    r_vec = np.array([r, 0.0, 0.0])
    v_vec = np.array([0.0, v_c*np.cos(i), v_c*np.sin(i)])
    return np.concatenate([r_vec, v_vec])


def coe_to_state(a, e, i_deg, raan_deg, omega_deg, theta_deg, mu=MU_EARTH):
    i, O, w, th = [np.radians(x) for x in (i_deg, raan_deg, omega_deg, theta_deg)]
    p   = a*(1 - e**2)
    r_m = p/(1 + e*np.cos(th))

    # Perifocal frame
    r_pf = r_m * np.array([np.cos(th), np.sin(th), 0.0])
    v_pf = np.sqrt(mu/p) * np.array([-np.sin(th), e + np.cos(th), 0.0])

    # Rotation matrix: perifocal -> ECI  (R3(-O)*R1(-i)*R3(-w))
    cO,sO = np.cos(O),np.sin(O)
    ci,si = np.cos(i),np.sin(i)
    cw,sw = np.cos(w),np.sin(w)
    Q = np.array([
        [ cO*cw - sO*sw*ci, -cO*sw - sO*cw*ci,  sO*si],
        [ sO*cw + cO*sw*ci, -sO*sw + cO*cw*ci, -cO*si],
        [ sw*si,              cw*si,              ci   ]
    ])
    return np.concatenate([Q @ r_pf, Q @ v_pf])


# Verify ISS-like orbit
s = circular_orbit_state(408, 51.6)
r0, v0 = s[:3], s[3:]
T_iss  = orbital_period(R_EARTH + 408, MU_EARTH)
print(f"ISS-like orbit:")
print(f"  r = {np.linalg.norm(r0):.2f} km   (expect {R_EARTH+408:.2f})")
print(f"  v = {np.linalg.norm(v0):.4f} km/s (expect {np.sqrt(MU_EARTH/(R_EARTH+408)):.4f})")
print(f"  e = {specific_energy(r0,v0,MU_EARTH):.4f} km2/s2 (< 0 => bound orbit)")
print(f"  T = {T_iss:.1f} s = {T_iss/60:.2f} min  (expect ~92 min)")


---
## 5. Simulate Three Representative Orbits

| Orbit | Altitude | Notable property |
|-------|----------|-----------------|
| **LEO** (ISS) | 408 km, i=51.6° | Fast, ~92 min period |
| **GEO** | 35,786 km, i=0° | Period = 1 sidereal day |
| **HEO** (Molniya-like) | $r_p$=600 km, $r_a$=39,000 km | High eccentricity |

The GEO period must be exactly **86,164 s**. If not, the integrator has a bug.


In [ ]:
# Orbit definitions
rp_heo  = R_EARTH + 600
ra_heo  = R_EARTH + 39_000
a_heo   = (rp_heo + ra_heo)/2
e_heo   = (ra_heo - rp_heo)/(ra_heo + rp_heo)

orbit_cfg = {
    "LEO (ISS)":    {"state0": circular_orbit_state(408,   51.6), "dt": 10,  "n_periods": 3,  "color": "#00bfff"},
    "GEO":          {"state0": circular_orbit_state(35786, 0.0),  "dt": 60,  "n_periods": 1,  "color": "#ff6b35"},
    "HEO (Molniya)":{"state0": coe_to_state(a_heo, e_heo, 63.4, 0, 270, 0),
                                                                   "dt": 10,  "n_periods": 1,  "color": "#7fff00"},
}

a_map = {
    "LEO (ISS)":     R_EARTH + 408,
    "GEO":           R_EARTH + 35786,
    "HEO (Molniya)": a_heo,
}

results = {}
print("Propagating orbits...")

for name, cfg in orbit_cfg.items():
    T   = orbital_period(a_map[name], MU_EARTH)
    tf  = T * cfg["n_periods"]
    traj = propagate(two_body_eom, cfg["state0"], (0, tf), cfg["dt"], MU_EARTH)

    eps = np.array([specific_energy(traj["r"][i], traj["v"][i], MU_EARTH)
                    for i in range(len(traj["t"]))])
    h   = np.array([np.linalg.norm(specific_ang_mom(traj["r"][i], traj["v"][i]))
                    for i in range(len(traj["t"]))])

    drift_e = abs((eps[-1]-eps[0])/eps[0])
    drift_h = abs((h[-1] -h[0]) /h[0])

    results[name] = {**cfg, "traj": traj, "T": T, "eps": eps, "h": h}

    print(f"  {name:<20}  steps={len(traj['t'])-1:>6,}  "
          f"de/e={drift_e:.1e}  dh/h={drift_h:.1e}")


---
## 6. 3D Visualization — ECI Frame

The **Earth-Centered Inertial (ECI)** frame:
- X axis: points toward the vernal equinox (does NOT rotate with Earth)
- Z axis: Earth's rotation axis (North Pole)
- Y axis: completes the right-hand system

Inclination visually tilts the orbit out of the equatorial (XY) plane.  
GEO sits in the equatorial plane. Molniya is tilted at 63.4° — a special angle  
that freezes the argument of perigee (no apsidal precession due to J₂). 


In [ ]:
def draw_earth(ax, n=30, alpha=0.25):
    u = np.linspace(0, 2*np.pi, n)
    v = np.linspace(0, np.pi,   n)
    x = R_EARTH * np.outer(np.cos(u), np.sin(v))
    y = R_EARTH * np.outer(np.sin(u), np.sin(v))
    z = R_EARTH * np.outer(np.ones(n), np.cos(v))
    ax.plot_surface(x, y, z, color='deepskyblue', alpha=alpha, linewidth=0)
    ax.plot_wireframe(x, y, z, color='royalblue', linewidth=0.25, alpha=0.35)

fig = plt.figure(figsize=(13, 9), facecolor='#0a0a1a')
ax  = fig.add_subplot(111, projection='3d')
ax.set_facecolor('#0a0a1a')

draw_earth(ax)

for name, data in results.items():
    r = data["traj"]["r"]
    ax.plot(r[:,0], r[:,1], r[:,2], color=data["color"],
            linewidth=1.4, label=name, alpha=0.9)
    ax.scatter(*r[0], color=data["color"], s=50, zorder=5)

# Equatorial plane reference
gr = 44000
gx = np.linspace(-gr, gr, 4)
GX, GY = np.meshgrid(gx, gx)
ax.plot_surface(GX, GY, np.zeros_like(GX), alpha=0.04, color='white')

lim = 44000
ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim); ax.set_zlim(-lim, lim)
ax.set_box_aspect([1,1,1])
ax.set_xlabel("X (km)  ->  Vernal equinox", color='white')
ax.set_ylabel("Y (km)", color='white')
ax.set_zlabel("Z (km)  ->  North Pole", color='white')
ax.set_title("Three Keplerian Orbits -- ECI Frame", color='white', fontsize=13, pad=15)
ax.tick_params(colors='gray')
ax.legend(loc='upper left', fontsize=10, facecolor='#1a1a2e', labelcolor='white')

plt.tight_layout()
plt.savefig("orbits_3d.png", dpi=150, facecolor='#0a0a1a', bbox_inches='tight')
plt.show()


---
## 7. Conservation Law Diagnostics

For a **pure two-body** simulation, energy $\varepsilon$ and $|\vec{h}|$ are mathematically constant.  
Any drift is **purely numerical error** from the integrator.

With RK4 at $dt=10$ s on LEO (~5400 s period):
- Expected relative drift: $\lesssim 10^{-8}$ per period
- Drift growing over many periods: normal (accumulation)
- Oscillating drift: round-off, not truncation error

> **Rule of thumb:** if $|\Delta\varepsilon/\varepsilon| > 10^{-4}$, your step size is too large.


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 7), facecolor='#0d0d1e')
fig.suptitle("Conservation Law Diagnostics -- RK4 Accuracy", color='white', fontsize=13)

for col, (name, data) in enumerate(results.items()):
    eps  = data["eps"]
    h    = data["h"]
    t_hr = data["traj"]["t"] / 3600

    eps_rel = (eps - eps[0]) / abs(eps[0])
    h_rel   = (h   - h[0])   / abs(h[0])

    for row, (arr, lbl) in enumerate([(eps_rel, "Energy drift"), (h_rel, "|h| drift")]):
        ax = axes[row, col]
        ax.set_facecolor('#111122')
        ax.plot(t_hr, arr, color=data["color"], linewidth=0.9)
        ax.axhline(0, color='white', linewidth=0.5, alpha=0.3, linestyle='--')
        ax.set_title(f"{name}\n{lbl}", fontsize=9, color='white')
        ax.tick_params(colors='gray')
        ax.yaxis.set_major_formatter(plt.FormatStrFormatter('%.1e'))
        if col == 0: ax.set_ylabel("Relative drift", color='gray', fontsize=9)
        if row == 1: ax.set_xlabel("Time (hours)", color='gray', fontsize=9)

plt.tight_layout()
plt.savefig("conservation.png", dpi=150, facecolor='#0d0d1e', bbox_inches='tight')
plt.show()

print("Drift summary:")
for name, data in results.items():
    eps = data["eps"]; h = data["h"]
    print(f"  {name:<22}  de/e={abs(eps[-1]-eps[0])/abs(eps[0]):.2e}  "
          f"dh/h={abs(h[-1]-h[0])/abs(h[0]):.2e}")


---
## 8. State Vector ↔ Orbital Elements

The inverse conversion (state vector → COEs) uses:

1. $\vec{h} = \vec{r} \times \vec{v}$ → inclination $i$
2. Node vector $\vec{N} = \hat{K} \times \vec{h}$ → RAAN $\Omega$  
3. Eccentricity vector $\vec{e}$ → eccentricity $e$ and argument of perigee $\omega$
4. True anomaly $\theta$ from angle between $\vec{e}$ and $\vec{r}$
5. Semi-major axis $a$ from specific energy: $a = -\mu/(2\varepsilon)$

**For a perfect two-body simulation, all COEs except $\theta$ must be constant.**  
When we add perturbations in Phase 3, you'll see them evolve.


In [ ]:
def state_to_coe(r_vec, v_vec, mu=MU_EARTH):
    r = np.linalg.norm(r_vec); v = np.linalg.norm(v_vec)
    h_vec = np.cross(r_vec, v_vec)
    h     = np.linalg.norm(h_vec)
    e_vec = ecc_vector(r_vec, v_vec, mu)
    e     = np.linalg.norm(e_vec)
    i     = np.degrees(np.arccos(np.clip(h_vec[2]/h, -1, 1)))

    K     = np.array([0.0, 0.0, 1.0])
    N_vec = np.cross(K, h_vec)
    N     = np.linalg.norm(N_vec)

    Omega = 0.0
    if N > 1e-10:
        Omega = np.degrees(np.arccos(np.clip(N_vec[0]/N, -1, 1)))
        if N_vec[1] < 0: Omega = 360 - Omega

    omega = 0.0
    if N > 1e-10 and e > 1e-10:
        omega = np.degrees(np.arccos(np.clip(np.dot(N_vec, e_vec)/(N*e), -1, 1)))
        if e_vec[2] < 0: omega = 360 - omega

    theta = 0.0
    if e > 1e-10:
        theta = np.degrees(np.arccos(np.clip(np.dot(e_vec, r_vec)/(e*r), -1, 1)))
        if np.dot(r_vec, v_vec) < 0: theta = 360 - theta

    eps_v = 0.5*v**2 - mu/r
    a     = -mu/(2*eps_v)

    return {"a": a, "e": e, "i": i, "Omega": Omega, "omega": omega, "theta": theta}


# Extract COEs over time for LEO
traj_leo = results["LEO (ISS)"]["traj"]
stride   = max(1, len(traj_leo["t"])//400)
idx      = np.arange(0, len(traj_leo["t"]), stride)
coe_t    = np.array([[ state_to_coe(traj_leo["r"][k], traj_leo["v"][k])["a"],
                        state_to_coe(traj_leo["r"][k], traj_leo["v"][k])["e"],
                        state_to_coe(traj_leo["r"][k], traj_leo["v"][k])["i"],
                        state_to_coe(traj_leo["r"][k], traj_leo["v"][k])["theta"] ]
                      for k in idx])

t_sub = traj_leo["t"][idx] / 3600

labels = ["Semi-major axis a (km)", "Eccentricity e", "Inclination i (deg)", "True Anomaly theta (deg)"]
colors = ["#00bfff", "#ff6b35", "#7fff00", "#ff00ff"]

fig, axes = plt.subplots(4, 1, figsize=(12, 10), sharex=True, facecolor='#0d0d1e')
fig.suptitle("Classical Orbital Elements over Time -- LEO (ISS)", color='white', fontsize=13)
for ax, arr, lbl, clr in zip(axes, coe_t.T, labels, colors):
    ax.plot(t_sub, arr, color=clr, linewidth=1)
    ax.set_ylabel(lbl, color='gray', fontsize=9)
    ax.set_facecolor('#111122')
    ax.tick_params(colors='gray')
axes[-1].set_xlabel("Time (hours)", color='gray')
plt.tight_layout()
plt.savefig("orbital_elements.png", dpi=150, facecolor='#0d0d1e', bbox_inches='tight')
plt.show()
print("COEs are constant (except theta) -- perfect Keplerian motion confirmed.")


---
## 9. Time Scaling Architecture

A fundamental design principle: **separate physics from display**.

The trajectory array is the *ground truth* — computed at full fidelity.  
Time scaling is implemented at the display layer by varying **frame stride** (index skip):

```
Real-time :  show every 1st  point  →  1x
100x speed:  show every 100th point →  100x  (same trajectory, fewer frames)
```

Changing `dt` changes *accuracy*, not speed. The table below makes this explicit.


In [ ]:
traj_leo = results["LEO (ISS)"]["traj"]
T_leo    = results["LEO (ISS)"]["T"]
dt_leo   = orbit_cfg["LEO (ISS)"]["dt"]
n_total  = len(traj_leo["t"])
fps      = 30

print(f"Trajectory: {n_total:,} points | dt={dt_leo}s | T={T_leo:.0f}s\n")
print(f"{'Speed':<18}  {'Stride':>8}  {'Frames':>8}  {'Video (s)':>10}  {'Orbit time (s)':>14}")
print("-"*60)
for label, stride in [("1x (real-time)", 1), ("10x", 10), ("100x", 100),
                       ("1000x", 1000), ("10000x", 10000)]:
    n_frames   = n_total // stride
    vid_s      = n_frames / fps
    orbit_time = n_total * dt_leo
    print(f"  {label:<16}  {stride:>8,}  {n_frames:>8,}  {vid_s:>10.1f}  {orbit_time:>14.0f}")

print("\nKey: the orbit_time (real physics) is the same for all rows.")
print("     Only the video duration (display) changes with stride.")


---
## 10. Export — CSV and JSON

Always export trajectories with **metadata**: integrator, step size, units, initial conditions.  
This lets you reload and analyze without rerunning the simulation.

In later phases we'll also export in Astropy ECSV and interface with real ephemerides.


In [ ]:
os.makedirs("exports", exist_ok=True)

def export_csv(name, traj, filename, stride=10):
    with open(filename, 'w', newline='') as f:
        w = csv.writer(f)
        w.writerow(["t_s","x_km","y_km","z_km","vx_kms","vy_kms","vz_kms","r_km","v_kms"])
        for i in range(0, len(traj["t"]), stride):
            r,v = traj["r"][i], traj["v"][i]
            w.writerow([f"{traj['t'][i]:.3f}",
                        f"{r[0]:.6f}", f"{r[1]:.6f}", f"{r[2]:.6f}",
                        f"{v[0]:.8f}", f"{v[1]:.8f}", f"{v[2]:.8f}",
                        f"{np.linalg.norm(r):.6f}", f"{np.linalg.norm(v):.8f}"])

def export_json(name, traj, cfg, filename, stride=10):
    idx  = range(0, len(traj["t"]), stride)
    data = {
        "metadata": {
            "orbit_name":  name,
            "integrator":  "RK4",
            "dt_s":        cfg["dt"],
            "mu_km3_s2":   MU_EARTH,
            "r0_km":       traj["r"][0].tolist(),
            "v0_kms":      traj["v"][0].tolist(),
            "frame":       "ECI",
            "units":       {"time":"s","position":"km","velocity":"km/s"},
            "exported_at": datetime.utcnow().isoformat()+"Z",
        },
        "t": [traj["t"][i] for i in idx],
        "r": [traj["r"][i].tolist() for i in idx],
        "v": [traj["v"][i].tolist() for i in idx],
    }
    with open(filename, 'w') as f:
        json.dump(data, f, indent=2)

print("Exporting trajectories...\n")
for name, data in results.items():
    safe = name.replace(" ","_").replace("(","").replace(")","").replace("/","_")
    traj = data["traj"]
    export_csv(name,  traj, f"exports/{safe}.csv",  stride=10)
    export_json(name, traj, orbit_cfg[name], f"exports/{safe}.json", stride=10)
    print(f"  {name}  ->  {safe}.csv  +  {safe}.json")

print("\nAll exports done!")


---
## 11. Summary & What's Next

### What you've built in Phase 1

| Component | Status |
|-----------|--------|
| Two-body EOM | ✅ |
| RK4 integrator (from scratch) | ✅ |
| COE <-> State vector conversion | ✅ |
| Conservation law validation | ✅ |
| 3D ECI orbit visualization | ✅ |
| Time-scaling architecture | ✅ |
| CSV + JSON export | ✅ |

### Physical reference values

| Quantity | Value |
|----------|-------|
| LEO orbital speed | ~7.7 km/s |
| ISS orbital period | ~92 min |
| GEO altitude | 35,786 km |
| GEO period | 86,164 s (1 sidereal day) |
| Escape velocity (surface) | 11.2 km/s = sqrt(2) × v_c |
| Molniya inclination | 63.4° (freezes apsidal precession) |

### Phase 2 — Orbital Maneuvers (next)
- Impulsive $\Delta v$ burns (instantaneous velocity change)
- Hohmann transfer (minimum energy orbit change)
- Bi-elliptic transfer
- Plane changes (most expensive maneuver)
- Full $\Delta v$ budgeting

### Phase 3 — Perturbations
- $J_2$ oblateness → RAAN precession + apsidal drift
- Atmospheric drag → orbital decay
- Third-body effects

---
*Orbital Mechanics Series -- Phase 1 complete*  
*Reference: `theory_01_two_body.md`*
